### **Setup**

In [2]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv
import sqlite3

load_dotenv()

print("✅ Imports successful!")
print("💾 Ready to add memory to agents!")

✅ Imports successful!
💾 Ready to add memory to agents!


### **Build a Simple Chatbot with Memory**

In [3]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)
# Node: Call LLM
def chat_node(state: ChatState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(ChatState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

memory = MemorySaver()
chatbot = builder.compile(checkpointer=memory)

print("✅ Chatbot with memory created!")

✅ Chatbot with memory created!


In [4]:
config = {"configurable": {"thread_id": "user-001"}}

print("=" * 60)
print("Conversation with Memory")
print("=" * 60)

result1 = chatbot.invoke(
    {"messages": [HumanMessage(content="My name is Alice and I love Python programming.")]},
    config
)
print(f"\n🧑 Alice: My name is Alice and I love Python programming.")
print(f"🤖 Bot: {result1['messages'][-1].content}")

Conversation with Memory

🧑 Alice: My name is Alice and I love Python programming.
🤖 Bot: Hello, Alice! 😊 It's great to meet a fellow Python enthusiast! Python is such a versatile and powerful programming language. What do you enjoy the most about Python? Are you working on any interesting projects or learning something new?


In [5]:
result2 = chatbot.invoke(
    {"messages": [HumanMessage(content="What's my name and what do I love?")]},
    config
)
print(f"\n🧑 Alice: What's my name and what do I love?")
print(f"🤖 Bot: {result2['messages'][-1].content}")


🧑 Alice: What's my name and what do I love?
🤖 Bot: Your name is **Alice**, and you love **Python programming**! 😊


In [7]:
result3 = chatbot.invoke(
    {"messages": [HumanMessage(content="Tell me a joke about it!")]},
    config
)
print(f"\n🧑 Alice: Tell me a joke about it!")
print(f"🤖 Bot: {result3['messages'][-1].content}")

print("\n" + "=" * 60)
print("✅ Memory working! Bot remembers context across turns.")
print("=" * 60)


🧑 Alice: Tell me a joke about it!
🤖 Bot: Of course, Alice! Here's another Python-related joke for you:

Why did the Python programmer break up with their partner?  
Because they kept *interpreting* things the wrong way! 🐍😄

Hope that gave you a chuckle! 😊

✅ Memory working! Bot remembers context across turns.


### **Multiple Conversations with Different Thread IDs**

In [11]:
config_bob = {"configurable": {"thread_id": "user-002"}}

result_bob = chatbot.invoke(
    {"messages": [HumanMessage(content="My name is Bob and I love JavaScript.")]},
    config_bob
)
print("🧑 Bob: My name is Bob and I love JavaScript.")
print(f"🤖 Bot: {result_bob['messages'][-1].content}")

🧑 Bob: My name is Bob and I love JavaScript.
🤖 Bot: Hi Bob! That's awesome to hear that you love JavaScript! 🚀 It's such a versatile and powerful language for both front-end and back-end development. Are you working on any cool JavaScript projects, or is there something specific you'd like to learn or discuss about it? 😊


In [10]:
result2 = chatbot.invoke(
    {"messages": [HumanMessage(content="What's my name and what do I love?")]},
    config
)
print(f"\n🧑 Alice: What's my name and what do I love?")
print(f"🤖 Bot: {result2['messages'][-1].content}")


🧑 Alice: What's my name and what do I love?
🤖 Bot: Your name is **Alice**, and you love **Python programming**! 🐍✨


In [12]:
result_bob2 = chatbot.invoke(
    {"messages": [HumanMessage(content="What do I love?")]},
    config_bob
)
print("\n🧑 Bob: What do I love?")
print(f"🤖 Bot: {result_bob2['messages'][-1].content}")


🧑 Bob: What do I love?
🤖 Bot: You love **JavaScript**! 🎉 Your enthusiasm for it is clear—whether you're building dynamic web apps, working with frameworks like React or Vue, or diving into Node.js for backend development, JavaScript is definitely your jam. Got a favorite feature of JavaScript, like async/await, closures, or ES6+ modules? Or maybe you just enjoy the endless possibilities it offers. Let me know! 😊


In [13]:
result_alice = chatbot.invoke(
    {"messages": [HumanMessage(content="What do I love again?")]},
    config  # Alice's thread_id
)
print("\n🧑 Alice: What do I love again?")
print(f"🤖 Bot: {result_alice['messages'][-1].content}")

print("\n✅ Separate threads maintain separate conversation contexts!")


🧑 Alice: What do I love again?
🤖 Bot: You love **Python programming**, Alice! 🐍💻 It's awesome that you're passionate about it! 😊

✅ Separate threads maintain separate conversation contexts!


### **SqliteSaver - Persistent File-Based Storage**

In [ ]:
db_path = "chatbot_memory.db"
conn = sqlite3.connect(db_path, check_same_thread=False)